# 01 — Data analysis
Inspect the locally supplied binary image dataset. No example data or metrics are bundled. The default path is `data/images/{real,synthetic}`.

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from src.preprocessing import IMAGE_EXTENSIONS
ROOT = Path('../data/images') if not Path('data/images').exists() else Path('data/images')
records = []
for label in ('real', 'synthetic'):
    folder = ROOT / label
    if not folder.is_dir():
        print(f'Missing {folder}; add the dataset as described in data/README.md')
        continue
    for path in folder.rglob('*'):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            try:
                with Image.open(path) as im:
                    records.append({'path': str(path), 'label': label, 'width': im.width, 'height': im.height, 'format': im.format})
            except Exception as exc:
                print(f'Skipping unreadable {path}: {exc}')
df = pd.DataFrame(records)
df.head()

In [ ]:
if df.empty:
    print('No readable image metadata to summarize.')
else:
    display(df.groupby('label').size().rename('image_count'))
    display(df.groupby('label')[['width', 'height']].describe())
    df['label'].value_counts().plot(kind='bar', title='Image count by class')
    plt.ylabel('Images'); plt.show()

## Inspect examples and observations
The next cell displays a small random-free sample (first files alphabetically). Use it to spot file/label mistakes; visual inspection does not certify authenticity. Document class imbalance, varying dimensions, corrupt files, and dataset source here after loading your own data.


In [ ]:
if not df.empty:
    sample = df.groupby('label', group_keys=False).head(3)
    fig, axes = plt.subplots(len(sample), 1, figsize=(8, 3 * len(sample)))
    axes = [axes] if len(sample) == 1 else axes
    for ax, row in zip(axes, sample.itertuples()):
        with Image.open(row.path) as im: ax.imshow(im.convert('RGB'))
        ax.set_title(f'{row.label}: {Path(row.path).name}'); ax.axis('off')
    plt.tight_layout(); plt.show()